# Condition C (Improved) — Fine-tuned CLIP ViT-L/14, Crops Only

Changes vs ablation-3:
- ViT-L/14 backbone (was ViT-B/32), unfreeze last 8 of 24 blocks
- K_VIEWS=4 hard negatives per item (was 2)
- SupConLoss temperature=0.05 (was 0.07)
- Layer-wise LR decay (LLRD=0.85) optimizer
- AMP (autocast + GradScaler) throughout
- EPOCHS=30, PATIENCE=5, eval every 2 epochs on 3000-query val subset
- Early stop on mAP@5 (was Recall@10)
- Alpha Query Expansion (α-QE) re-ranking at final eval
- Text embeddings pre-computed once outside seed loop

In [ ]:
!pip install git+https://github.com/openai/CLIP.git -q
!pip install faiss-cpu -q
print('Done.')

import os, json, math, random, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
import faiss
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict
from IPython.display import FileLink

In [ ]:
SEEDS  = [2023006, 2023008, 2023524, 2023552]
ALPHAS = [0.7, 0.85]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

In [ ]:
ROOT     = '/kaggle/input/datasets/dedeepyaavancha/deepfashion-in-shop-clothes-retrieval-benchmark/In-shop Clothes Retrieval Benchmark'
SAVE_DIR = '/kaggle/working/condition_c_v2'
os.makedirs(SAVE_DIR, exist_ok=True)

print('ROOT exists    :', os.path.exists(ROOT))

In [ ]:
# ---- Load splits ----
eval_path = os.path.join(ROOT, 'Eval', 'list_eval_partition.txt')
df = pd.read_csv(
    eval_path, sep=r'\s+', skiprows=2, header=None,
    names=['image_name', 'item_id', 'split']
)

# ---- Load bounding boxes ----
bbox_path = os.path.join(ROOT, 'Anno', 'list_bbox_inshop.txt')
df_bbox = pd.read_csv(
    bbox_path, sep=r'\s+', skiprows=2, header=None,
    names=['image_name', 'clothes_type', 'pose_type', 'x1', 'y1', 'x2', 'y2']
)

# ---- Merge ----
df = df.merge(df_bbox, on='image_name', how='left')
df['image_path'] = df['image_name'].apply(lambda x: os.path.join(ROOT, x))

# ---- Splits ----
df_train   = df[df['split'] == 'train'].reset_index(drop=True)
df_query   = df[df['split'] == 'query'].reset_index(drop=True)
df_gallery = df[df['split'] == 'gallery'].reset_index(drop=True)

gallery_ids     = df_gallery['item_id'].tolist()
query_ids       = df_query['item_id'].tolist()
gallery_ids_arr = np.array(gallery_ids)
gallery_image_names = df_gallery['image_name'].tolist()

with open(f"{SAVE_DIR}/gallery_ids.json", "w") as f:
    json.dump(gallery_image_names, f)

with open(f"{SAVE_DIR}/query_ids.json", "w") as f:
    json.dump(df_query['image_name'].tolist(), f)

print(f'Train   : {len(df_train)}')
print(f'Query   : {len(df_query)}')
print(f'Gallery : {len(df_gallery)}')
print(f'Unique train items: {df_train["item_id"].nunique()}')

In [ ]:
def get_model():
    # ViT-L/14: 24 transformer blocks, 307M params vs 151M for ViT-B/32
    clip_model, preprocess = clip.load('ViT-L/14', device=device)
    clip_model = clip_model.float()

    # Freeze everything
    for param in clip_model.parameters():
        param.requires_grad = False

    # Unfreeze last 8 transformer blocks (was 4)
    vision_blocks = clip_model.visual.transformer.resblocks
    n_blocks = len(vision_blocks)

    for block in vision_blocks[n_blocks - 8:]:
        for param in block.parameters():
            param.requires_grad = True

    # Unfreeze final layers
    for param in clip_model.visual.ln_post.parameters():
        param.requires_grad = True

    if clip_model.visual.proj is not None:
        clip_model.visual.proj.requires_grad = True

    trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in clip_model.parameters())

    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
    print('CLIP ViT-L/14 ready.')

    return clip_model, preprocess

In [ ]:
N_ITEMS = 8    # 8 items × 4 views = batch 32
K_VIEWS = 4

from torchvision import transforms as T

# Applied to PIL crop before CLIP preprocess — adds stochasticity so each
# epoch sees a different view of the same crop (prevents SupCon memorization)
TRAIN_AUG = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
])


class BalancedSampler(Sampler):
    def __init__(self, df, n_items, k_views, seed):
        self.n_items = n_items
        self.k_views = k_views
        self.seed    = seed

        self.id_to_idxs = defaultdict(list)
        for i, row in df.reset_index(drop=True).iterrows():
            self.id_to_idxs[row['item_id']].append(i)

        self.valid_ids = [
            k for k, v in self.id_to_idxs.items()
            if len(v) >= k_views
        ]

        print(f'Valid items (>={k_views} views): {len(self.valid_ids)}')

    def __iter__(self):
        rng = random.Random(self.seed)

        ids = rng.sample(self.valid_ids, len(self.valid_ids))
        batch = []

        for item_id in ids:
            views = rng.sample(self.id_to_idxs[item_id], self.k_views)
            batch.extend(views)

            if len(batch) >= self.n_items * self.k_views:
                yield from batch
                batch = []

    def __len__(self):
        return len(self.valid_ids) * self.k_views


class CropDataset(Dataset):
    def __init__(self, df, preprocess, augment=False):
        self.df         = df.reset_index(drop=True)
        self.preprocess = preprocess
        self.augment    = augment

        unique_ids     = sorted(self.df['item_id'].unique())
        self.id_to_int = {v: i for i, v in enumerate(unique_ids)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row['image_path']).convert('RGB')
        img = img.crop((row['x1'], row['y1'], row['x2'], row['y2']))
        if img.size[0] < 10 or img.size[1] < 10:
            img = Image.open(row['image_path']).convert('RGB')

        if self.augment:
            img = TRAIN_AUG(img)   # augment before CLIP preprocess

        return self.preprocess(img), self.id_to_int[row['item_id']]

In [ ]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.05):   # was 0.07 — sharper distribution
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        B        = features.shape[0]
        features = F.normalize(features, dim=1)

        sim = torch.matmul(features, features.T) / self.temperature

        labels   = labels.unsqueeze(1)
        pos_mask = (labels == labels.T).float()
        pos_mask.fill_diagonal_(0)

        sim = sim - torch.max(sim, dim=1, keepdim=True)[0].detach()

        eye     = torch.eye(B, device=features.device)
        exp_sim = torch.exp(sim) * (1 - eye)

        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

        n_pos = pos_mask.sum(dim=1)
        loss  = -(pos_mask * log_prob).sum(dim=1)
        loss  = loss[n_pos > 0] / n_pos[n_pos > 0]

        return loss.mean()


criterion = SupConLoss(temperature=0.05)
print('SupCon loss ready (temperature=0.05).')

In [ ]:
def compute_metrics(query_ids, indices, gallery_ids_arr, Ks=[5, 10, 15]):
    results = {}

    for K in Ks:
        recalls, ndcgs, aps = [], [], []

        for q_idx, q_id in enumerate(query_ids):
            retrieved = gallery_ids_arr[indices[q_idx, :K]]
            relevant  = (retrieved == q_id)

            recalls.append(1.0 if relevant.any() else 0.0)

            dcg  = sum(rel / math.log2(r + 2) for r, rel in enumerate(relevant))
            idcg = sum(1.0 / math.log2(r + 2)
                       for r in range(min(int(relevant.sum()), K)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

            hits, prec = 0, 0.0
            for r, rel in enumerate(relevant):
                if rel:
                    hits += 1
                    prec += hits / (r + 1)

            aps.append(
                prec / min(int(relevant.sum()), K)
                if relevant.sum() > 0 else 0.0
            )

        results[K] = {
            'Recall': np.mean(recalls),
            'NDCG'  : np.mean(ndcgs),
            'mAP'   : np.mean(aps)
        }

    return results


def print_metrics(metrics, label):
    print(f'\n{"="*58}')
    print(f'  {label}')
    print(f'{"="*58}')
    print(f'{"K":<6} {"Recall@K":<12} {"NDCG@K":<12} {"mAP@K":<12}')
    print('-'*58)

    for K, v in metrics.items():
        print(f'@{K:<5} {v["Recall"]:.4f}      {v["NDCG"]:.4f}      {v["mAP"]:.4f}')

    print('='*58)


print('Metrics ready.')

In [ ]:
def embed_crops(df, clip_model, preprocess, batch_size=512):  # was 256
    clip_model.eval()

    all_embs, all_ids = [], []

    with torch.no_grad(), torch.amp.autocast('cuda'):
        for start in range(0, len(df), batch_size):
            batch = df.iloc[start:start+batch_size]

            imgs = []
            for _, row in batch.iterrows():
                img = Image.open(row['image_path']).convert('RGB')
                img = img.crop((row['x1'], row['y1'],
                                row['x2'], row['y2']))
                imgs.append(preprocess(img))

            imgs = torch.stack(imgs).to(device)

            embs = clip_model.encode_image(imgs)
            embs = embs / embs.norm(dim=-1, keepdim=True)

            all_embs.append(embs.cpu().float().numpy())
            all_ids.extend(batch['item_id'].tolist())

    return np.vstack(all_embs), all_ids


print('Embedding function ready.')

In [ ]:
# ---- Load pre-generated BLIP captions from Kaggle dataset ----
# Dataset: BLIP_Captions
# Files:   gallery_captions_crop.json, query_captions_crop.json
# Kaggle mounts input datasets under /kaggle/input/<slug>/

def _find_file(filename):
    """Walk /kaggle/input to find a file by name."""
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return os.path.join(root, filename)
    return None

gallery_cap_src = _find_file('gallery_captions_crop.json')
query_cap_src   = _find_file('query_captions_crop.json')

print('Gallery caps:', gallery_cap_src)
print('Query caps  :', query_cap_src)

assert gallery_cap_src, 'gallery_captions_crop.json not found under /kaggle/input'
assert query_cap_src,   'query_captions_crop.json not found under /kaggle/input'

with open(gallery_cap_src) as f:
    gallery_captions_crop = json.load(f)
with open(query_cap_src) as f:
    query_captions_crop = json.load(f)

# Copy to SAVE_DIR so the zip is self-contained
with open(f'{SAVE_DIR}/gallery_caps.json', 'w') as f:
    json.dump(gallery_captions_crop, f)
with open(f'{SAVE_DIR}/query_caps.json', 'w') as f:
    json.dump(query_captions_crop, f)

print(f'Gallery captions : {len(gallery_captions_crop)}')
print(f'Query captions   : {len(query_captions_crop)}')
assert len(gallery_captions_crop) == len(df_gallery), 'Caption/gallery length mismatch'
assert len(query_captions_crop)   == len(df_query),   'Caption/query length mismatch'

In [ ]:
def encode_text(captions, clip_model, batch_size=256):
    clip_model.eval()

    all_embs = []

    with torch.no_grad(), torch.amp.autocast('cuda'):
        for start in range(0, len(captions), batch_size):
            tokens = clip.tokenize(
                captions[start:start+batch_size], truncate=True
            ).to(device)

            embs = clip_model.encode_text(tokens)
            embs = embs / embs.norm(dim=-1, keepdim=True)

            all_embs.append(embs.cpu().float().numpy())

    return np.vstack(all_embs)

In [ ]:
def fuse(img_embs, txt_embs, alpha):
    fused = alpha * img_embs + (1 - alpha) * txt_embs
    return fused / np.linalg.norm(fused, axis=1, keepdims=True)

def run_retrieval(g_embs, q_embs, label, q_ids=None):
    if q_ids is None:
        q_ids = query_ids
    index = faiss.IndexFlatIP(g_embs.shape[1])
    index.add(g_embs.astype(np.float32))
    sims, idxs = index.search(q_embs.astype(np.float32), 15)
    m = compute_metrics(q_ids, idxs, gallery_ids_arr)
    print_metrics(m, label)
    return m, sims, idxs

In [ ]:
def alpha_query_expansion(q_embs, g_embs, alpha_exp=3.0, top_k=10):
    """
    Standard α-QE (Radenovic et al.):
    weights normalized to sum=1, so neighbor_avg is a proper weighted mean
    of unit vectors (magnitude ~0.5-0.9). q + neighbor_avg has magnitude
    ~1.5-1.9 and the original query keeps ~50-70% influence after renorm.
    """
    index = faiss.IndexFlatIP(g_embs.shape[1])
    index.add(g_embs.astype(np.float32))

    sims, idxs = index.search(q_embs.astype(np.float32), top_k)

    q_expanded = np.zeros_like(q_embs)
    for i in range(len(q_embs)):
        w = np.clip(sims[i], 0, None) ** alpha_exp
        w /= (w.sum() + 1e-8)                          # normalize: sum=1
        neighbor_avg = (w[:, None] * g_embs[idxs[i]]).sum(axis=0)
        q_exp = q_embs[i] + neighbor_avg               # magnitude ~1.5-1.9
        q_expanded[i] = q_exp / (np.linalg.norm(q_exp) + 1e-8)

    _, reranked_idxs = index.search(q_expanded.astype(np.float32), g_embs.shape[0])
    return reranked_idxs


print('Alpha-QE ready.')

In [ ]:
EPOCHS       = 30
LR           = 1e-5
PATIENCE     = 5
EVAL_EVERY   = 4
VAL_QUERIES  = 3000
LLRD         = 0.85
WARMUP_EPOCHS = 2     # linear warmup before cosine decay

# Pre-compute text embeddings ONCE (text encoder stays frozen across seeds)
print('Pre-computing text embeddings...')
_tmp_model, _ = get_model()
gallery_txt = encode_text(gallery_captions_crop, _tmp_model)
query_txt   = encode_text(query_captions_crop,   _tmp_model)
del _tmp_model
torch.cuda.empty_cache()
print(f'Gallery text: {gallery_txt.shape}, Query text: {query_txt.shape}')

all_seed_results = []
history_all = []

for seed in SEEDS:
    print(f"\n================ SEED {seed} ================")

    set_seed(seed)
    torch.cuda.empty_cache()

    # ---- MODEL ----
    clip_model, preprocess = get_model()

    # ---- VALIDATION SUBSET ----
    rng_val   = np.random.RandomState(seed)
    val_idx   = rng_val.choice(len(df_query), VAL_QUERIES, replace=False)
    df_q_val  = df_query.iloc[val_idx].reset_index(drop=True)
    q_ids_val = df_q_val['item_id'].tolist()
    query_txt_val = query_txt[val_idx]

    # ---- DATA ---- (augment=True prevents SupCon from memorizing training items)
    train_dataset = CropDataset(df_train, preprocess, augment=True)
    sampler       = BalancedSampler(df_train, N_ITEMS, K_VIEWS, seed=seed)

    train_loader  = DataLoader(
        train_dataset,
        batch_size=N_ITEMS * K_VIEWS,
        sampler=sampler,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        drop_last=True
    )

    # ---- LAYER-WISE LR DECAY OPTIMIZER ----
    vision_blocks    = clip_model.visual.transformer.resblocks
    n_blocks         = len(vision_blocks)
    trainable_blocks = vision_blocks[n_blocks - 8:]

    param_groups = []
    for depth, block in enumerate(reversed(list(trainable_blocks))):
        block_lr = LR * (LLRD ** depth)
        param_groups.append({'params': list(block.parameters()), 'lr': block_lr})

    head_params = list(clip_model.visual.ln_post.parameters())
    if clip_model.visual.proj is not None:
        head_params.append(clip_model.visual.proj)
    param_groups.insert(0, {'params': head_params, 'lr': LR})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=3e-4)  # was 1e-4

    # Warmup for WARMUP_EPOCHS, then cosine decay to 0
    total_steps  = EPOCHS * len(train_loader)
    warmup_steps = WARMUP_EPOCHS * len(train_loader)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    scaler = torch.amp.GradScaler('cuda')

    best_map      = 0.0
    epochs_no_imp = 0
    history       = []

    # ---- TRAIN ----
    for epoch in range(1, EPOCHS + 1):
        clip_model.train()

        epoch_loss = 0.0
        n_batches  = 0

        for imgs, labels in train_loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                embs = clip_model.encode_image(imgs)
                embs = embs / embs.norm(dim=-1, keepdim=True)
                loss = criterion(embs, labels)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, clip_model.parameters()), 1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            epoch_loss += loss.item()
            n_batches  += 1

        avg_loss = epoch_loss / n_batches
        cur_lr   = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {cur_lr:.2e}')

        # ---- EVAL every EVAL_EVERY epochs ----
        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            print('  Evaluating on val subset...')

            g_emb, _     = embed_crops(df_gallery, clip_model, preprocess)
            q_emb_val, _ = embed_crops(df_q_val,   clip_model, preprocess)

            val_map = 0.0
            for alpha in ALPHAS:
                alpha_key = round(alpha, 2)
                gf = fuse(g_emb,     gallery_txt,   alpha)
                qf = fuse(q_emb_val, query_txt_val, alpha)
                m, _, _ = run_retrieval(
                    gf, qf,
                    f'Val Seed {seed} | Ep {epoch} | α={alpha_key}',
                    q_ids=q_ids_val
                )
                if alpha_key == 0.7:
                    val_map = m[5]['mAP']

            history.append({
                'epoch'              : epoch,
                'loss'               : avg_loss,
                'mAP@5_val_alpha0.7' : val_map
            })

            if val_map > best_map:
                best_map      = val_map
                epochs_no_imp = 0
                torch.save(
                    clip_model.state_dict(),
                    f'{SAVE_DIR}/best_clip_seed{seed}.pt'
                )
                print(f'  Best model (mAP@5={best_map:.4f})')
            else:
                epochs_no_imp += 1
                print(f'  No improvement ({epochs_no_imp}/{PATIENCE})')
                if epochs_no_imp >= PATIENCE:
                    print(f'  Early stopping at epoch {epoch}')
                    break

    history_all.append({"seed": seed, "history": history})

    # ---- FINAL EVAL WITH BEST MODEL ----
    print('Loading best checkpoint...')
    clip_model.load_state_dict(
        torch.load(f'{SAVE_DIR}/best_clip_seed{seed}.pt')
    )
    clip_model.eval()

    g_emb_best, _ = embed_crops(df_gallery, clip_model, preprocess)
    q_emb_best, _ = embed_crops(df_query,   clip_model, preprocess)

    seed_res = {"seed": seed, "alphas": {}}

    for alpha in ALPHAS:
        alpha_key = round(alpha, 2)

        gf = fuse(g_emb_best, gallery_txt, alpha)
        qf = fuse(q_emb_best, query_txt,   alpha)

        m, _, _ = run_retrieval(gf, qf, f'Seed {seed} FINAL α={alpha_key}')

        rr_idxs = alpha_query_expansion(qf, gf, alpha_exp=3.0, top_k=10)
        m_qe    = compute_metrics(query_ids, rr_idxs, gallery_ids_arr)
        print_metrics(m_qe, f'Seed {seed} α-QE FINAL α={alpha_key}')

        seed_res["alphas"][alpha_key] = {'standard': m, 'alpha_qe': m_qe}

        if alpha_key == 0.7 and seed == 2023008:
            np.save(f"{SAVE_DIR}/gallery_fused.npy",     gf)
            np.save(f"{SAVE_DIR}/query_fused.npy",       qf)
            np.save(f"{SAVE_DIR}/gallery_embs_crop.npy", g_emb_best)
            np.save(f"{SAVE_DIR}/query_embs_crop.npy",   q_emb_best)
            np.save(f"{SAVE_DIR}/gallery_text_embs.npy", gallery_txt)
            np.save(f"{SAVE_DIR}/query_text_embs.npy",   query_txt)

    all_seed_results.append(seed_res)

In [ ]:
flat_results = []

for res in all_seed_results:
    row = {"seed": res["seed"]}

    for alpha_key, metric_dict in res["alphas"].items():
        for tag, metrics in metric_dict.items():   # 'standard', 'alpha_qe'
            for K in [5, 10, 15]:
                row[f"alpha_{alpha_key}_{tag}_R@{K}"]    = metrics[K]["Recall"]
                row[f"alpha_{alpha_key}_{tag}_NDCG@{K}"] = metrics[K]["NDCG"]
                row[f"alpha_{alpha_key}_{tag}_mAP@{K}"]  = metrics[K]["mAP"]

    flat_results.append(row)

df_res = pd.DataFrame(flat_results)

mean_res = df_res.drop(columns=["seed"]).mean()
std_res  = df_res.drop(columns=["seed"]).std()

print("\n===== FINAL MEAN ± STD =====")
for col in mean_res.index:
    print(f"{col}: {mean_res[col]:.4f} ± {std_res[col]:.4f}")

df_res.to_csv(f"{SAVE_DIR}/results_per_seed.csv", index=False)

summary_df = pd.DataFrame({
    "metric": mean_res.index,
    "mean"  : mean_res.values,
    "std"   : std_res.values
})
summary_df.to_csv(f"{SAVE_DIR}/results_summary.csv", index=False)
print("Saved CSVs.")

In [ ]:
with open(f'{SAVE_DIR}/history_all_seeds.json', 'w') as f:
    json.dump(history_all, f, indent=4)

zip_path = f'{SAVE_DIR}/condition_c_v2_multiseed.zip'

IMPORTANT_FILES = [
    "gallery_caps.json",
    "query_caps.json",
    "gallery_ids.json",
    "query_ids.json",
    "gallery_fused.npy",
    "query_fused.npy",
    "gallery_embs_crop.npy",
    "query_embs_crop.npy",
    "gallery_text_embs.npy",
    "query_text_embs.npy",
    "results_summary.csv",
    "results_per_seed.csv",
    "best_clip_seed2023008.pt",
    "history_all_seeds.json"
]

with zipfile.ZipFile(zip_path, 'w') as z:
    for fname in IMPORTANT_FILES:
        path = f'{SAVE_DIR}/{fname}'
        if os.path.exists(path):
            z.write(path, fname)

print("Zip created.")
FileLink(zip_path)